# 03 — Transform Raw Data into Staging

## Banking Reporting Platform

This notebook is the first **data-quality gate** in the pipeline.

The raw layer preserved every source row exactly as received. This notebook now applies the approved business rules and separates those rows into:

```text
raw
 │
 ├── valid rows   → staging
 │
 └── invalid rows → audit.rejected_records
```

The staging layer contains:

```text
staging.customers
staging.accounts
staging.customer_accounts
staging.channels
staging.transactions
```

The audit layer contains:

```text
audit.rejected_records
```

### Transformation policy

This project uses a simple deterministic rule for duplicate business keys:

> **First valid occurrence wins; later duplicate occurrences are rejected.**

That applies to:

- `customer_id`
- `account_id`
- (`customer_id`, `account_id`)
- `transaction_id`

This avoids silently keeping multiple versions of the same source business key while still preserving the rejected row in the audit layer.


## 1. Imports and Project Configuration

The notebook uses:

- `pandas` for validation and transformation
- `psycopg` for PostgreSQL reads and writes
- PostgreSQL `COPY` for efficient staging loads
- `Jsonb` for rejected-record payloads


In [10]:
from pathlib import Path
import json

import pandas as pd
import psycopg
from psycopg import sql
from psycopg.types.json import Jsonb
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected a data/raw directory "
        "in the current directory or one of its parents."
    )


def read_env_file(path: Path) -> dict[str, str]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} was not found. Create .env from .env.example before continuing."
        )

    values = {}

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")

    return values


PROJECT_ROOT = find_project_root(Path.cwd())
ENV_PATH = PROJECT_ROOT / ".env"

env = read_env_file(ENV_PATH)

DB_CONFIG = {
    "host": "localhost",
    "port": int(env["POSTGRES_PORT"]),
    "dbname": env["POSTGRES_DB"],
    "user": env["POSTGRES_USER"],
    "password": env["POSTGRES_PASSWORD"],
}


def get_connection():
    return psycopg.connect(**DB_CONFIG)


print(f"Project root: {PROJECT_ROOT}")
print(
    "PostgreSQL target:",
    f"{DB_CONFIG['user']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}",
)


Project root: C:\Users\Admin\Projects\banking-reporting-platform
PostgreSQL target: banking_admin@localhost:5432/analytics


## 2. Confirm the Raw Layer Is Available

Notebook 02 must have completed successfully before continuing.


In [11]:
RAW_TABLES = [
    "customers",
    "accounts",
    "customer_accounts",
    "transactions",
]

raw_counts = []

with get_connection() as conn:
    with conn.cursor() as cur:
        for table_name in RAW_TABLES:
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM raw.{}").format(
                    sql.Identifier(table_name)
                )
            )
            raw_counts.append(
                {
                    "raw_table": f"raw.{table_name}",
                    "rows": cur.fetchone()[0],
                }
            )

raw_counts = pd.DataFrame(raw_counts)
display(raw_counts)

if (raw_counts["rows"] == 0).any():
    raise RuntimeError(
        "At least one raw table is empty. Run 02_ingest_raw.ipynb first."
    )


,raw_table,rows
0,raw.customers,1003
1,raw.accounts,1253
2,raw.customer_accounts,1355
3,raw.transactions,50010


## 3. Create the Staging and Audit Structures

These definitions implement the approved physical model.

The staging tables use proper PostgreSQL data types and constraints. The raw tables intentionally did not.


In [12]:
STAGING_DDL = '''
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS audit;

CREATE TABLE IF NOT EXISTS staging.customers (
    customer_id VARCHAR(20) PRIMARY KEY,
    customer_since_date DATE NOT NULL,
    customer_status VARCHAR(10) NOT NULL
        CHECK (customer_status IN ('ACTIVE', 'INACTIVE')),
    loaded_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS staging.accounts (
    account_id VARCHAR(20) PRIMARY KEY,
    account_type VARCHAR(20) NOT NULL
        CHECK (account_type IN ('TRANSACTION', 'SAVINGS')),
    account_status VARCHAR(10) NOT NULL
        CHECK (account_status IN ('ACTIVE', 'DORMANT', 'CLOSED')),
    opened_date DATE NOT NULL,
    closed_date DATE,
    loaded_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    CHECK (closed_date IS NULL OR closed_date >= opened_date),
    CHECK (account_status <> 'CLOSED' OR closed_date IS NOT NULL)
);

CREATE TABLE IF NOT EXISTS staging.customer_accounts (
    customer_id VARCHAR(20) NOT NULL
        REFERENCES staging.customers(customer_id),
    account_id VARCHAR(20) NOT NULL
        REFERENCES staging.accounts(account_id),
    holder_role VARCHAR(10) NOT NULL
        CHECK (holder_role IN ('PRIMARY', 'JOINT')),
    loaded_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    PRIMARY KEY (customer_id, account_id)
);

CREATE TABLE IF NOT EXISTS staging.channels (
    channel_code VARCHAR(10) PRIMARY KEY,
    channel_name VARCHAR(50) NOT NULL
);

CREATE TABLE IF NOT EXISTS staging.transactions (
    transaction_id VARCHAR(30) PRIMARY KEY,
    account_id VARCHAR(20) NOT NULL
        REFERENCES staging.accounts(account_id),
    channel_code VARCHAR(10) NOT NULL
        REFERENCES staging.channels(channel_code),
    transaction_timestamp TIMESTAMP NOT NULL,
    transaction_type VARCHAR(20) NOT NULL
        CHECK (transaction_type IN ('PURCHASE', 'WITHDRAWAL', 'DEPOSIT', 'TRANSFER')),
    amount NUMERIC(18, 2) NOT NULL
        CHECK (amount > 0),
    currency_code CHAR(3) NOT NULL
        CHECK (currency_code = 'ZAR'),
    status VARCHAR(15) NOT NULL
        CHECK (status IN ('SUCCESSFUL', 'FAILED', 'REVERSED')),
    loaded_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS audit.rejected_records (
    rejection_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_name VARCHAR(50) NOT NULL,
    record_key TEXT,
    rejection_reason TEXT NOT NULL,
    raw_payload JSONB NOT NULL,
    rejected_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS idx_staging_transactions_account_id
    ON staging.transactions(account_id);

CREATE INDEX IF NOT EXISTS idx_staging_transactions_timestamp
    ON staging.transactions(transaction_timestamp);

CREATE INDEX IF NOT EXISTS idx_staging_transactions_channel_code
    ON staging.transactions(channel_code);
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(STAGING_DDL)

print("Staging and audit structures are ready.")


Staging and audit structures are ready.


## 4. Full-Refresh the Staging and Audit Layers

This notebook is rerunnable.

Because the current project treats each run as a complete source-extract refresh, the existing staging rows and rejected-record audit rows are cleared before transformation.


In [13]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            TRUNCATE TABLE
                staging.transactions,
                staging.customer_accounts,
                staging.channels,
                staging.accounts,
                staging.customers,
                audit.rejected_records
            RESTART IDENTITY;
            '''
        )

print("Existing staging and rejected-record rows removed.")


Existing staging and rejected-record rows removed.


## 5. Read Raw Tables into DataFrames

The raw ingestion metadata is kept while we validate because it is useful inside rejected-record payloads.


In [14]:
def read_table(schema_name: str, table_name: str) -> pd.DataFrame:
    query = sql.SQL("SELECT * FROM {}.{}").format(
        sql.Identifier(schema_name),
        sql.Identifier(table_name),
    )

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            columns = [description.name for description in cur.description]
            rows = cur.fetchall()

    return pd.DataFrame(rows, columns=columns)


raw_customers = read_table("raw", "customers")
raw_accounts = read_table("raw", "accounts")
raw_customer_accounts = read_table("raw", "customer_accounts")
raw_transactions = read_table("raw", "transactions")

print(f"raw.customers:         {len(raw_customers):,}")
print(f"raw.accounts:          {len(raw_accounts):,}")
print(f"raw.customer_accounts: {len(raw_customer_accounts):,}")
print(f"raw.transactions:      {len(raw_transactions):,}")


raw.customers:         1,003
raw.accounts:          1,253
raw.customer_accounts: 1,355
raw.transactions:      50,010


## 6. Validation Helpers

Each invalid source row creates **one** audit record.

If a row violates several rules, all reasons are combined into one semicolon-separated `rejection_reason`.


In [15]:
def blank_mask(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.strip().eq("")


def normalise_text(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.strip().str.upper()


def add_reason(reason_lists, mask: pd.Series, reason: str) -> None:
    for position in mask[mask].index:
        reason_lists[position].append(reason)


def row_to_jsonable_dict(row: pd.Series) -> dict:
    payload = {}

    for key, value in row.items():
        if pd.isna(value):
            payload[key] = None
        elif hasattr(value, "isoformat"):
            payload[key] = value.isoformat()
        else:
            payload[key] = str(value)

    return payload


def build_rejected_rows(
    source_name: str,
    raw_df: pd.DataFrame,
    reasons: list[list[str]],
    record_key_builder,
) -> list[dict]:
    rejected = []

    for position, reason_list in enumerate(reasons):
        if not reason_list:
            continue

        row = raw_df.iloc[position]

        rejected.append(
            {
                "source_name": source_name,
                "record_key": record_key_builder(row),
                "rejection_reason": "; ".join(dict.fromkeys(reason_list)),
                "raw_payload": row_to_jsonable_dict(row),
            }
        )

    return rejected


def copy_dataframe(
    dataframe: pd.DataFrame,
    schema_name: str,
    table_name: str,
    columns: list[str],
) -> int:
    if dataframe.empty:
        return 0

    copy_statement = sql.SQL("COPY {}.{} ({}) FROM STDIN").format(
        sql.Identifier(schema_name),
        sql.Identifier(table_name),
        sql.SQL(", ").join(sql.Identifier(column) for column in columns),
    )

    rows_written = 0

    with get_connection() as conn:
        with conn.cursor() as cur:
            with cur.copy(copy_statement) as copy:
                for row in dataframe[columns].itertuples(index=False, name=None):
                    # PostgreSQL expects SQL NULL for missing values.
                    # pandas may represent missing dates as NaT and missing
                    # scalar values as NaN/pd.NA, so normalize all of them
                    # to Python None before passing rows to psycopg COPY.
                    clean_row = tuple(
                        None if pd.isna(value) else value
                        for value in row
                    )

                    copy.write_row(clean_row)
                    rows_written += 1

    return rows_written


all_rejected = []


## 7. Transform Customers

Rules:

- `customer_id` is required
- first occurrence of `customer_id` wins
- `customer_since_date` must parse as a date
- `customer_status` must be `ACTIVE` or `INACTIVE`


In [16]:
customers = raw_customers.copy().reset_index(drop=True)
customer_reasons = [[] for _ in range(len(customers))]

customer_id_clean = customers["customer_id"].fillna("").astype(str).str.strip()
customer_status_clean = normalise_text(customers["customer_status"])
customer_since_parsed = pd.to_datetime(
    customers["customer_since_date"],
    errors="coerce",
)

add_reason(
    customer_reasons,
    customer_id_clean.eq(""),
    "customer_id is blank",
)

add_reason(
    customer_reasons,
    customer_id_clean.duplicated(keep="first"),
    "duplicate customer_id; first occurrence retained",
)

add_reason(
    customer_reasons,
    customer_since_parsed.isna(),
    "customer_since_date is not a valid date",
)

add_reason(
    customer_reasons,
    ~customer_status_clean.isin({"ACTIVE", "INACTIVE"}),
    "customer_status is not an allowed value",
)

customer_valid_mask = pd.Series(
    [len(reasons) == 0 for reasons in customer_reasons]
)

staging_customers = pd.DataFrame(
    {
        "customer_id": customer_id_clean[customer_valid_mask],
        "customer_since_date": customer_since_parsed[customer_valid_mask].dt.date,
        "customer_status": customer_status_clean[customer_valid_mask],
    }
).reset_index(drop=True)

customer_rejected = build_rejected_rows(
    "customers",
    customers,
    customer_reasons,
    lambda row: str(row.get("customer_id", "")),
)

all_rejected.extend(customer_rejected)

display(
    pd.DataFrame(
        [
            {
                "raw_rows": len(customers),
                "valid_rows": len(staging_customers),
                "rejected_rows": len(customer_rejected),
            }
        ]
    )
)


,raw_rows,valid_rows,rejected_rows
0,1003,1000,3


## 8. Transform Accounts

Rules:

- `account_id` is required
- first occurrence of `account_id` wins
- account type must be `TRANSACTION` or `SAVINGS`
- account status must be `ACTIVE`, `DORMANT`, or `CLOSED`
- `opened_date` must be valid
- non-blank `closed_date` must be valid
- `closed_date` cannot be before `opened_date`
- a `CLOSED` account must have `closed_date`


In [17]:
accounts = raw_accounts.copy().reset_index(drop=True)
account_reasons = [[] for _ in range(len(accounts))]

account_id_clean = accounts["account_id"].fillna("").astype(str).str.strip()
account_type_clean = normalise_text(accounts["account_type"])
account_status_clean = normalise_text(accounts["account_status"])

opened_parsed = pd.to_datetime(
    accounts["opened_date"],
    errors="coerce",
)

closed_source_blank = blank_mask(accounts["closed_date"])

closed_parsed = pd.to_datetime(
    accounts["closed_date"].replace("", pd.NA),
    errors="coerce",
)

add_reason(
    account_reasons,
    account_id_clean.eq(""),
    "account_id is blank",
)

add_reason(
    account_reasons,
    account_id_clean.duplicated(keep="first"),
    "duplicate account_id; first occurrence retained",
)

add_reason(
    account_reasons,
    ~account_type_clean.isin({"TRANSACTION", "SAVINGS"}),
    "account_type is not an allowed value",
)

add_reason(
    account_reasons,
    ~account_status_clean.isin({"ACTIVE", "DORMANT", "CLOSED"}),
    "account_status is not an allowed value",
)

add_reason(
    account_reasons,
    opened_parsed.isna(),
    "opened_date is not a valid date",
)

add_reason(
    account_reasons,
    (~closed_source_blank) & closed_parsed.isna(),
    "closed_date is not a valid date",
)

add_reason(
    account_reasons,
    opened_parsed.notna()
    & closed_parsed.notna()
    & (closed_parsed < opened_parsed),
    "closed_date is earlier than opened_date",
)

add_reason(
    account_reasons,
    account_status_clean.eq("CLOSED") & closed_source_blank,
    "CLOSED account is missing closed_date",
)

account_valid_mask = pd.Series(
    [len(reasons) == 0 for reasons in account_reasons]
)

staging_accounts = pd.DataFrame(
    {
        "account_id": account_id_clean[account_valid_mask],
        "account_type": account_type_clean[account_valid_mask],
        "account_status": account_status_clean[account_valid_mask],
        "opened_date": opened_parsed[account_valid_mask].dt.date,
        "closed_date": closed_parsed[account_valid_mask].dt.date,
    }
).reset_index(drop=True)

account_rejected = build_rejected_rows(
    "accounts",
    accounts,
    account_reasons,
    lambda row: str(row.get("account_id", "")),
)

all_rejected.extend(account_rejected)

display(
    pd.DataFrame(
        [
            {
                "raw_rows": len(accounts),
                "valid_rows": len(staging_accounts),
                "rejected_rows": len(account_rejected),
            }
        ]
    )
)


,raw_rows,valid_rows,rejected_rows
0,1253,1250,3


## 9. Load Valid Customers and Accounts

The relationship and transaction validations depend on these accepted business keys, so the parent entities are loaded first.


In [18]:
customers_loaded = copy_dataframe(
    staging_customers,
    "staging",
    "customers",
    [
        "customer_id",
        "customer_since_date",
        "customer_status",
    ],
)

accounts_loaded = copy_dataframe(
    staging_accounts,
    "staging",
    "accounts",
    [
        "account_id",
        "account_type",
        "account_status",
        "opened_date",
        "closed_date",
    ],
)

print(f"Customers loaded: {customers_loaded:,}")
print(f"Accounts loaded:  {accounts_loaded:,}")


Customers loaded: 1,000
Accounts loaded:  1,250


## 10. Transform Customer–Account Relationships

Rules:

- customer and account IDs are required
- first occurrence of a customer-account pair wins
- holder role must be `PRIMARY` or `JOINT`
- referenced customer must exist in accepted staging customers
- referenced account must exist in accepted staging accounts
- only one `PRIMARY` holder is retained for an account
- every accepted account must end with exactly one `PRIMARY` holder and at least one holder


In [19]:
customer_accounts = raw_customer_accounts.copy().reset_index(drop=True)
relationship_reasons = [[] for _ in range(len(customer_accounts))]

ca_customer_id = customer_accounts["customer_id"].fillna("").astype(str).str.strip()
ca_account_id = customer_accounts["account_id"].fillna("").astype(str).str.strip()
holder_role_clean = normalise_text(customer_accounts["holder_role"])

accepted_customer_ids = set(staging_customers["customer_id"])
accepted_account_ids = set(staging_accounts["account_id"])

add_reason(
    relationship_reasons,
    ca_customer_id.eq(""),
    "customer_id is blank",
)

add_reason(
    relationship_reasons,
    ca_account_id.eq(""),
    "account_id is blank",
)

relationship_key_df = pd.DataFrame(
    {
        "customer_id": ca_customer_id,
        "account_id": ca_account_id,
    }
)

add_reason(
    relationship_reasons,
    relationship_key_df.duplicated(
        subset=["customer_id", "account_id"],
        keep="first",
    ),
    "duplicate customer-account relationship; first occurrence retained",
)

add_reason(
    relationship_reasons,
    ~holder_role_clean.isin({"PRIMARY", "JOINT"}),
    "holder_role is not an allowed value",
)

add_reason(
    relationship_reasons,
    ~ca_customer_id.isin(accepted_customer_ids),
    "customer_id does not reference an accepted staging customer",
)

add_reason(
    relationship_reasons,
    ~ca_account_id.isin(accepted_account_ids),
    "account_id does not reference an accepted staging account",
)

# Reject later PRIMARY rows on an account after the first otherwise-valid PRIMARY.
primary_seen = set()

for position in range(len(customer_accounts)):
    if relationship_reasons[position]:
        continue

    if holder_role_clean.iloc[position] != "PRIMARY":
        continue

    account_id = ca_account_id.iloc[position]

    if account_id in primary_seen:
        relationship_reasons[position].append(
            "account already has a PRIMARY holder; first PRIMARY retained"
        )
    else:
        primary_seen.add(account_id)

relationship_valid_mask = pd.Series(
    [len(reasons) == 0 for reasons in relationship_reasons]
)

staging_customer_accounts = pd.DataFrame(
    {
        "customer_id": ca_customer_id[relationship_valid_mask],
        "account_id": ca_account_id[relationship_valid_mask],
        "holder_role": holder_role_clean[relationship_valid_mask],
    }
).reset_index(drop=True)

relationship_rejected = build_rejected_rows(
    "customer_accounts",
    customer_accounts,
    relationship_reasons,
    lambda row: (
        f"{row.get('customer_id', '')}|{row.get('account_id', '')}"
    ),
)

all_rejected.extend(relationship_rejected)

# Post-validation of the accepted relationship set.
holder_counts = (
    staging_customer_accounts.groupby("account_id")
    .size()
    .reindex(staging_accounts["account_id"], fill_value=0)
)

primary_counts = (
    staging_customer_accounts.assign(
        is_primary=staging_customer_accounts["holder_role"].eq("PRIMARY").astype(int)
    )
    .groupby("account_id")["is_primary"]
    .sum()
    .reindex(staging_accounts["account_id"], fill_value=0)
)

if (holder_counts < 1).any():
    bad_accounts = holder_counts[holder_counts < 1].index.tolist()
    raise RuntimeError(
        f"Accepted accounts without a valid holder: {bad_accounts[:10]}"
    )

if (primary_counts != 1).any():
    bad_accounts = primary_counts[primary_counts != 1].index.tolist()
    raise RuntimeError(
        f"Accepted accounts without exactly one PRIMARY holder: {bad_accounts[:10]}"
    )

display(
    pd.DataFrame(
        [
            {
                "raw_rows": len(customer_accounts),
                "valid_rows": len(staging_customer_accounts),
                "rejected_rows": len(relationship_rejected),
            }
        ]
    )
)


,raw_rows,valid_rows,rejected_rows
0,1355,1350,5


## 11. Load Valid Customer–Account Relationships


In [20]:
relationships_loaded = copy_dataframe(
    staging_customer_accounts,
    "staging",
    "customer_accounts",
    [
        "customer_id",
        "account_id",
        "holder_role",
    ],
)

print(f"Customer-account relationships loaded: {relationships_loaded:,}")


Customer-account relationships loaded: 1,350


## 12. Build the Channel Reference

The modelling documents define the valid channel codes as:

```text
APP  → Mobile App
ATM  → ATM
CARD → Card
```

The staging channel table is populated only for valid channel codes actually present in the raw transaction extract.


In [21]:
CHANNEL_NAMES = {
    "APP": "Mobile App",
    "ATM": "ATM",
    "CARD": "Card",
}

raw_channel_codes = set(normalise_text(raw_transactions["channel_code"]))

staging_channels = pd.DataFrame(
    [
        {
            "channel_code": code,
            "channel_name": CHANNEL_NAMES[code],
        }
        for code in CHANNEL_NAMES
        if code in raw_channel_codes
    ]
).sort_values("channel_code").reset_index(drop=True)

channels_loaded = copy_dataframe(
    staging_channels,
    "staging",
    "channels",
    [
        "channel_code",
        "channel_name",
    ],
)

display(staging_channels)
print(f"Channels loaded: {channels_loaded:,}")


,channel_code,channel_name
0,APP,Mobile App
1,ATM,ATM
2,CARD,Card


Channels loaded: 3


## 13. Transform Transactions

Rules:

- transaction ID and account ID are required
- first occurrence of `transaction_id` wins
- account must exist in accepted staging accounts
- timestamp must be valid
- transaction type must be allowed
- channel must be one of the accepted channel codes
- amount must be numeric and greater than zero
- currency must be `ZAR`
- status must be `SUCCESSFUL`, `FAILED`, or `REVERSED`
- transaction date cannot be before account opening
- transaction date cannot be after account closure


In [22]:
transactions = raw_transactions.copy().reset_index(drop=True)
transaction_reasons = [[] for _ in range(len(transactions))]

tx_id = transactions["transaction_id"].fillna("").astype(str).str.strip()
tx_account_id = transactions["account_id"].fillna("").astype(str).str.strip()
tx_type = normalise_text(transactions["transaction_type"])
tx_channel = normalise_text(transactions["channel_code"])
tx_currency = normalise_text(transactions["currency_code"])
tx_status = normalise_text(transactions["status"])

tx_timestamp = pd.to_datetime(
    transactions["transaction_timestamp"],
    errors="coerce",
)

tx_amount = pd.to_numeric(
    transactions["amount"],
    errors="coerce",
)

add_reason(
    transaction_reasons,
    tx_id.eq(""),
    "transaction_id is blank",
)

add_reason(
    transaction_reasons,
    tx_id.duplicated(keep="first"),
    "duplicate transaction_id; first occurrence retained",
)

add_reason(
    transaction_reasons,
    tx_account_id.eq(""),
    "account_id is blank",
)

add_reason(
    transaction_reasons,
    ~tx_account_id.isin(accepted_account_ids),
    "account_id does not reference an accepted staging account",
)

add_reason(
    transaction_reasons,
    tx_timestamp.isna(),
    "transaction_timestamp is not a valid timestamp",
)

add_reason(
    transaction_reasons,
    ~tx_type.isin({"PURCHASE", "WITHDRAWAL", "DEPOSIT", "TRANSFER"}),
    "transaction_type is not an allowed value",
)

accepted_channel_codes = set(staging_channels["channel_code"])

add_reason(
    transaction_reasons,
    ~tx_channel.isin(accepted_channel_codes),
    "channel_code is not an allowed channel",
)

add_reason(
    transaction_reasons,
    tx_amount.isna(),
    "amount is not numeric",
)

add_reason(
    transaction_reasons,
    tx_amount.notna() & (tx_amount <= 0),
    "amount must be greater than zero",
)

add_reason(
    transaction_reasons,
    ~tx_currency.eq("ZAR"),
    "currency_code must be ZAR",
)

add_reason(
    transaction_reasons,
    ~tx_status.isin({"SUCCESSFUL", "FAILED", "REVERSED"}),
    "status is not an allowed value",
)

# Account lifecycle checks use the accepted staging accounts.
account_lifecycle = staging_accounts[
    [
        "account_id",
        "opened_date",
        "closed_date",
    ]
].copy()

account_lifecycle["opened_date"] = pd.to_datetime(
    account_lifecycle["opened_date"]
)
account_lifecycle["closed_date"] = pd.to_datetime(
    account_lifecycle["closed_date"]
)

lifecycle_lookup = account_lifecycle.set_index("account_id")

for position in range(len(transactions)):
    account_id = tx_account_id.iloc[position]
    timestamp = tx_timestamp.iloc[position]

    if account_id not in lifecycle_lookup.index or pd.isna(timestamp):
        continue

    account = lifecycle_lookup.loc[account_id]
    transaction_date = timestamp.normalize()
    opened_date = account["opened_date"]
    closed_date = account["closed_date"]

    if transaction_date < opened_date:
        transaction_reasons[position].append(
            "transaction occurred before account opened_date"
        )

    if pd.notna(closed_date) and transaction_date > closed_date:
        transaction_reasons[position].append(
            "transaction occurred after account closed_date"
        )

transaction_valid_mask = pd.Series(
    [len(reasons) == 0 for reasons in transaction_reasons]
)

staging_transactions = pd.DataFrame(
    {
        "transaction_id": tx_id[transaction_valid_mask],
        "account_id": tx_account_id[transaction_valid_mask],
        "channel_code": tx_channel[transaction_valid_mask],
        "transaction_timestamp": tx_timestamp[transaction_valid_mask],
        "transaction_type": tx_type[transaction_valid_mask],
        "amount": tx_amount[transaction_valid_mask].round(2),
        "currency_code": tx_currency[transaction_valid_mask],
        "status": tx_status[transaction_valid_mask],
    }
).reset_index(drop=True)

transaction_rejected = build_rejected_rows(
    "transactions",
    transactions,
    transaction_reasons,
    lambda row: str(row.get("transaction_id", "")),
)

all_rejected.extend(transaction_rejected)

display(
    pd.DataFrame(
        [
            {
                "raw_rows": len(transactions),
                "valid_rows": len(staging_transactions),
                "rejected_rows": len(transaction_rejected),
            }
        ]
    )
)


,raw_rows,valid_rows,rejected_rows
0,50010,50000,10


## 14. Load Valid Transactions


In [23]:
transactions_loaded = copy_dataframe(
    staging_transactions,
    "staging",
    "transactions",
    [
        "transaction_id",
        "account_id",
        "channel_code",
        "transaction_timestamp",
        "transaction_type",
        "amount",
        "currency_code",
        "status",
    ],
)

print(f"Transactions loaded: {transactions_loaded:,}")


Transactions loaded: 50,000


## 15. Write Rejected Rows to the Audit Layer

Every invalid raw row receives one audit record with:

- source entity
- business key
- combined rejection reason
- original raw payload
- rejection timestamp


In [24]:
AUDIT_INSERT = '''
INSERT INTO audit.rejected_records (
    source_name,
    record_key,
    rejection_reason,
    raw_payload
)
VALUES (%s, %s, %s, %s);
'''

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.executemany(
            AUDIT_INSERT,
            [
                (
                    record["source_name"],
                    record["record_key"],
                    record["rejection_reason"],
                    Jsonb(record["raw_payload"]),
                )
                for record in all_rejected
            ],
        )

print(f"Rejected records written to audit: {len(all_rejected):,}")


Rejected records written to audit: 21


## 16. Reconcile Raw, Staging, and Audit Counts

For each source entity:

```text
raw rows = accepted staging rows + rejected audit rows
```

This is the central control for this transformation step.


In [25]:
staging_counts = {
    "customers": len(staging_customers),
    "accounts": len(staging_accounts),
    "customer_accounts": len(staging_customer_accounts),
    "transactions": len(staging_transactions),
}

raw_entity_counts = {
    "customers": len(raw_customers),
    "accounts": len(raw_accounts),
    "customer_accounts": len(raw_customer_accounts),
    "transactions": len(raw_transactions),
}

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT source_name, COUNT(*)
            FROM audit.rejected_records
            GROUP BY source_name
            ORDER BY source_name;
            '''
        )
        rejected_counts = dict(cur.fetchall())

reconciliation_rows = []

for source_name in raw_entity_counts:
    raw_rows = raw_entity_counts[source_name]
    valid_rows = staging_counts[source_name]
    rejected_rows = rejected_counts.get(source_name, 0)

    reconciliation_rows.append(
        {
            "source": source_name,
            "raw_rows": raw_rows,
            "staging_rows": valid_rows,
            "rejected_rows": rejected_rows,
            "difference": raw_rows - valid_rows - rejected_rows,
            "reconciled": raw_rows == valid_rows + rejected_rows,
        }
    )

reconciliation = pd.DataFrame(reconciliation_rows)
display(reconciliation)

if not reconciliation["reconciled"].all():
    raise RuntimeError("Raw-to-staging reconciliation failed.")

print("All source entities reconcile.")


,source,raw_rows,staging_rows,rejected_rows,difference,reconciled
0,customers,1003,1000,3,0,True
1,accounts,1253,1250,3,0,True
2,customer_accounts,1355,1350,5,0,True
3,transactions,50010,50000,10,0,True


All source entities reconcile.


## 17. Review Rejection Reasons

This gives us an audit summary without manually scanning every rejected row.


In [26]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT
                source_name,
                rejection_reason,
                COUNT(*) AS rejected_rows
            FROM audit.rejected_records
            GROUP BY source_name, rejection_reason
            ORDER BY source_name, rejected_rows DESC, rejection_reason;
            '''
        )

        columns = [description.name for description in cur.description]
        rejection_summary = pd.DataFrame(cur.fetchall(), columns=columns)

display(rejection_summary)


,source_name,rejection_reason,rejected_rows
0,accounts,account_type is not an allowed value,1
1,accounts,closed_date is earlier than opened_date,1
2,accounts,duplicate account_id; first occurrence retained,1
3,customer_accounts,account already has a PRIMARY holder; first PR...,1
4,customer_accounts,account_id does not reference an accepted stag...,1
5,customer_accounts,customer_id does not reference an accepted sta...,1
6,customer_accounts,duplicate customer-account relationship; first...,1
7,customer_accounts,holder_role is not an allowed value,1
8,customers,customer_since_date is not a valid date,1
9,customers,customer_status is not an allowed value,1


## 18. Final Staging Data-Quality Checks

The database constraints already protect the staging tables, but we also run explicit checks for the business rules that are not fully expressed as standard table constraints.


In [27]:
quality_checks = []

with get_connection() as conn:
    with conn.cursor() as cur:
        checks = {
            "customers have unique IDs": '''
                SELECT COUNT(*) - COUNT(DISTINCT customer_id)
                FROM staging.customers;
            ''',
            "accounts have unique IDs": '''
                SELECT COUNT(*) - COUNT(DISTINCT account_id)
                FROM staging.accounts;
            ''',
            "transactions have unique IDs": '''
                SELECT COUNT(*) - COUNT(DISTINCT transaction_id)
                FROM staging.transactions;
            ''',
            "every account has at least one holder": '''
                SELECT COUNT(*)
                FROM staging.accounts a
                LEFT JOIN staging.customer_accounts ca
                    ON ca.account_id = a.account_id
                WHERE ca.account_id IS NULL;
            ''',
            "every account has exactly one PRIMARY holder": '''
                SELECT COUNT(*)
                FROM (
                    SELECT
                        a.account_id,
                        COUNT(*) FILTER (
                            WHERE ca.holder_role = 'PRIMARY'
                        ) AS primary_holders
                    FROM staging.accounts a
                    LEFT JOIN staging.customer_accounts ca
                        ON ca.account_id = a.account_id
                    GROUP BY a.account_id
                    HAVING COUNT(*) FILTER (
                        WHERE ca.holder_role = 'PRIMARY'
                    ) <> 1
                ) problems;
            ''',
            "transactions do not occur after closure": '''
                SELECT COUNT(*)
                FROM staging.transactions t
                JOIN staging.accounts a
                    ON a.account_id = t.account_id
                WHERE a.closed_date IS NOT NULL
                  AND t.transaction_timestamp::date > a.closed_date;
            ''',
            "transactions do not occur before opening": '''
                SELECT COUNT(*)
                FROM staging.transactions t
                JOIN staging.accounts a
                    ON a.account_id = t.account_id
                WHERE t.transaction_timestamp::date < a.opened_date;
            ''',
        }

        for check_name, query in checks.items():
            cur.execute(query)
            failed_rows = cur.fetchone()[0]

            quality_checks.append(
                {
                    "check": check_name,
                    "failed_rows": failed_rows,
                    "passed": failed_rows == 0,
                }
            )

quality_checks = pd.DataFrame(quality_checks)
display(quality_checks)

if not quality_checks["passed"].all():
    raise RuntimeError("One or more final staging quality checks failed.")

print("All final staging quality checks passed.")


,check,failed_rows,passed
0,customers have unique IDs,0,True
1,accounts have unique IDs,0,True
2,transactions have unique IDs,0,True
3,every account has at least one holder,0,True
4,every account has exactly one PRIMARY holder,0,True
5,transactions do not occur after closure,0,True
6,transactions do not occur before opening,0,True


All final staging quality checks passed.


## 19. Inspect the Final Staging Layer

At this point, the staging tables should contain only typed, validated records.


In [28]:
final_counts = []

with get_connection() as conn:
    with conn.cursor() as cur:
        for table_name in [
            "customers",
            "accounts",
            "customer_accounts",
            "channels",
            "transactions",
        ]:
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM staging.{}").format(
                    sql.Identifier(table_name)
                )
            )

            final_counts.append(
                {
                    "staging_table": f"staging.{table_name}",
                    "rows": cur.fetchone()[0],
                }
            )

        cur.execute("SELECT COUNT(*) FROM audit.rejected_records;")
        rejected_total = cur.fetchone()[0]

display(pd.DataFrame(final_counts))
print(f"Total rejected records: {rejected_total:,}")


,staging_table,rows
0,staging.customers,1000
1,staging.accounts,1250
2,staging.customer_accounts,1350
3,staging.channels,3
4,staging.transactions,50000


Total rejected records: 21


## 20. Staging Transformation Conclusion

We now have a proper quality boundary between raw source data and trusted analytical data.

```text
raw
 │
 ├── invalid → audit.rejected_records
 │
 └── valid
       ↓
    staging
```

The staging layer now provides:

- validated business keys
- typed dates and timestamps
- typed numeric transaction amounts
- controlled domain values
- valid customer-account relationships
- valid account references
- valid transaction channels
- account-lifecycle validation
- deterministic duplicate handling
- complete rejection auditing
- raw-to-staging reconciliation

### Next step

`04_build_marts.ipynb`

That notebook will transform the validated staging layer into the dimensional reporting model:

```text
staging
   ↓
dim_customer
dim_account
dim_channel
dim_date
bridge_account_customer
fact_transactions
```

It will also implement our agreed **SCD Type 1** behaviour for `dim_customer` and `dim_account`.
